# Лабораторная работа 9. Вариант 1
**Тема.** Open-shop задачи

## **Задача 1**
$$
O_2 | \space | C_{max}
$$ 
Задачи типа **open-shop** характеризуются свободным порядком выполнения работ. Решение здесь находится через LAPT [1, 8.1]. Суть LAPT:
> Приоритет отдается работам с наибольшей суммарной длительностью выполнения на всех станках

Для каждой работы считается $P_j$:
$$
P_j = p_{j1}+p_{j2}
$$
после чего работы упорядочиваются по $P_j$

In [1]:
jobs = {
    'J1': {'p1': 1, 'p2': 13},
    'J2': {'p1': 10, 'p2': 12},
    'J3': {'p1': 17, 'p2': 9},
    'J4': {'p1': 12, 'p2': 17},
    'J5': {'p1': 11, 'p2': 3}
}

In [2]:
done = {j_id: {1: False, 2: False} for j_id in jobs}
m_free = {1: 0, 2: 0}
j_busy_until = {j_id: 0 for j_id in jobs}

schedule = []
total_ops = len(jobs) * 2
completed_ops = 0

current_time = 0

In [3]:
while completed_ops < total_ops:
    for m_id in [1, 2]:
        if m_free[m_id] <= current_time:
            other_m = 2 if m_id == 1 else 1
            candidates = []
            for j_id, times in jobs.items():
                if not done[j_id][m_id] and j_busy_until[j_id] <= current_time:
                    weight = times[f'p{other_m}']
                    candidates.append((weight, j_id))
            
            if candidates:
                candidates.sort(key=lambda x: x[0], reverse=True)
                _, best_j = candidates[0]
                
                p_curr = jobs[best_j][f'p{m_id}']
                start = current_time
                end = start + p_curr
                
                m_free[m_id] = end
                j_busy_until[best_j] = end
                done[best_j][m_id] = True
                completed_ops += 1
                schedule.append((best_j, m_id, start, end))
    
    current_time = min(m_free.values()) if any(not d[1] or not d[2] for d in done.values()) else current_time + 1
    if current_time > 1000: break

In [4]:
plan, c_max = schedule, max(m_free.values())

print(f"{'Работа':<7} | {'Станок':<7} | {'Начало':<7} | {'Конец'}")
print("-" * 40)
for j, m, s, e in sorted(plan, key=lambda x: x[2]):
    print(f"{j:<7} | M{m:<6} | {s:<7} | {e}")
print("-" * 40)
print(f"Итоговый C_max: {c_max}")

Работа  | Станок  | Начало  | Конец
----------------------------------------
J4      | M1      | 0       | 12
J3      | M2      | 0       | 9
J5      | M2      | 9       | 12
J1      | M1      | 12      | 13
J4      | M2      | 12      | 29
J2      | M1      | 13      | 23
J3      | M1      | 23      | 40
J2      | M2      | 29      | 41
J5      | M1      | 40      | 51
J1      | M2      | 41      | 54
----------------------------------------
Итоговый C_max: 54


$$
O_3 | \space | C_{max}
$$ 
Алгоритм: Longest Total Remaining Processing on Other Machines first rule (8.1)

In [5]:
jobs = {
    'J1': {1: 1, 2: 13, 3: 6},
    'J2': {1: 10, 2: 12, 3: 18},
    'J3': {1: 17, 2: 9, 3: 13},
    'J4': {1: 12, 2: 17, 3: 2},
    'J5': {1: 11, 2: 3, 3: 5}
}

In [6]:
all_machines = sorted(list(set(m for j in jobs.values() for m in j.keys())))
done = {j_id: {m_id: (m_id not in jobs[j_id]) for m_id in all_machines} for j_id in jobs}

m_free = {m_id: 0 for m_id in all_machines}
j_busy_until = {j_id: 0 for j_id in jobs}

schedule = []
total_ops = sum(len(j_ops) for j_ops in jobs.values())
completed_ops = 0
current_time = 0

In [7]:
while completed_ops < total_ops:
    progress = False
    available_machines = [m for m in all_machines if m_free[m] <= current_time]
    
    for m_id in available_machines:
        candidates = []
        for j_id, p_times in jobs.items():
            if m_id in p_times and not done[j_id][m_id] and j_busy_until[j_id] <= current_time:
                weight = sum(p_times[m] for m in p_times if m != m_id and not done[j_id][m])
                candidates.append((weight, j_id))
        
        if candidates:
            candidates.sort(key=lambda x: x[0], reverse=True)
            _, best_j = candidates[0]
            
            duration = jobs[best_j][m_id]
            start = current_time
            end = start + duration
            
            m_free[m_id] = end
            j_busy_until[best_j] = end
            done[best_j][m_id] = True
            completed_ops += 1
            schedule.append({'j': best_j, 'm': m_id, 's': start, 'e': end})
            progress = True

    events = [t for t in list(m_free.values()) + list(j_busy_until.values()) if t > current_time]
    if events:
        current_time = min(events)
    else:
        if completed_ops < total_ops: current_time += 1
        else: break

In [8]:
plan, final_cmax = schedule, max(m_free.values())

print(f"{'Работа':<7} | {'Станок':<7} | {'Начало':<7} | {'Конец'}")
print("-" * 40)
for op in sorted(plan, key=lambda x: (x['s'], x['m'])):
    print(f"{op['j']:<7} | M{op['m']:<6} | {op['s']:<7} | {op['e']}")
print("-" * 40)
print(f"C_max (LTRP-OM): {final_cmax}")

Работа  | Станок  | Начало  | Конец
----------------------------------------
J2      | M1      | 0       | 10
J3      | M2      | 0       | 9
J4      | M3      | 0       | 2
J1      | M3      | 2       | 8
J5      | M3      | 8       | 13
J4      | M2      | 9       | 26
J1      | M1      | 10      | 11
J3      | M1      | 11      | 28
J2      | M3      | 13      | 31
J5      | M2      | 26      | 29
J4      | M1      | 28      | 40
J1      | M2      | 29      | 42
J3      | M3      | 31      | 44
J5      | M1      | 40      | 51
J2      | M2      | 42      | 54
----------------------------------------
C_max (LTRP-OM): 54


$$
O_3 | prmp | C_{max}
$$ 
Алгоритм: Гонсалеса-Сахни или Биркгофа — фон Неймана

In [9]:
jobs = {
    'J1': {1: 1, 2: 13, 3: 6},
    'J2': {1: 10, 2: 12, 3: 18},
    'J3': {1: 17, 2: 9, 3: 13},
    'J4': {1: 12, 2: 17, 3: 2},
    'J5': {1: 11, 2: 3, 3: 5}
}

In [10]:
m_loads = {1:0, 2:0, 3:0}
j_loads = {j: sum(ops.values()) for j, ops in jobs.items()}
for j in jobs:
    for m, p in jobs[j].items():
        m_loads[m] += p
        
c_max_target = max(max(m_loads.values()), max(j_loads.values()))

In [11]:
c_max_min = c_max_target
print(f"Минимально возможный C_max с прерываниями: {c_max_min}")

Минимально возможный C_max с прерываниями: 54
